In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile
import ipywidgets as W
from ipywidgets import interact, widgets, fixed

# Add src to path if running from notebooks folder
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
from render import localization_function, render_marker, pool_image, build_comps, render_cell, N_POOLS, POOL_NAMES, field
from tape import Tape

%matplotlib inline

# 1) Synthetic cell generation

In [ ]:
SEED=None

K=20
SIZE = 201
TILE = 256
N_CAND = 1500

# Draw 3DTape. For the later optimization, we want to draw the random numbers once, and then use them for all parameter combinations. This is to avoid the random numbers changing when we change parameters.
# instead of a 2d size, now its a 3d volume
VOL=(128, 128, 128)
# Ca 10 pixels per ym VERIFY!
UM_PER_VOX = 10


tape = Tape(seed=SEED, size=SIZE, K=K)
tape.draw(tile = TILE, n_cand = N_CAND, Pool = N_POOLS)

In [ ]:
from dataclasses import dataclass

FLUOROPHORES = {
    "DAPI": 0.461, "FITC": 0.519, "PE": 0.578, "APC": 0.660
}

# Optics: 
@dataclass(frozen=True)
class Optics:
    """Known Physical properties of the detector (Macsima) and the experiment."""
    um_per_px: float = 0.325 
    focal_um: float = 0.0
    
    # VERIFY
    na: float = 0.75 #or 0.45 
    wavelength_um: float = 0.530 
    
    n_immersion: float = 1.0
    
    @property
    def tan_theta(self):
        return float(np.tan(np.arcsin(np.clip(self.na / self.n_immersion, 0.0, 0.999))))